# Suitable for constructing enzyme-bound models

In [1]:
import cobra
import datetime 
import pandas as pd
import subprocess
import re
import sys
sys.path.append(r'./script/')
from ECMpy_function import *

D:\pycharm\ecCG827\.venv\Lib\site-packages\bioservices\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
model_file = "iCZ870_CGXII.xml"
modelseed_met_file = './data/modelseed_compounds.tsv'
Determine_suitable_ecGEM(model_file,modelseed_met_file)

Set parameter Username
Set parameter LicenseID to value 2723056
Academic license - for non-commercial use only - expires 2026-10-16


No objective coefficients in model. Unclear what should be optimized


('Yes', ['⭐Suitable for constructing enzyme-bound models⭐.'])

# get_reaction_kcat_using_DLKcat

## for UniKP

In [13]:
DLouputdf_file = 'get_kcat_mw_by_UniKP/unikp_prediction_results.csv'
DL_reaction_kact_mw_file='./get_kcat_mw_by_UniKP/reaction_kcat_MW.csv'

## for catpred

In [15]:
DLouputdf_file = 'get_kcat_mw_by_catpred/catpred_output.tsv'
DL_reaction_kact_mw_file='./get_kcat_mw_by_catpred/reaction_kcat_MW.csv'

# for catapro

In [17]:
DLouputdf_file = 'get_kcat_mw_by_catapro/catapro_prediction_noH.tsv'
DL_reaction_kact_mw_file='./get_kcat_mw_by_catapro/reaction_kcat_MW.csv'

## 1.1 input

In [3]:
dlkcat_folder = "./get_kcat_mw_by_DLKcat/"
create_file(dlkcat_folder)
# input files
sbml_path = "iCZ870_CGXII.xml"
sequence_file = './data/Cg14067_sequence.xlsx'
# output files
gene_subnum_path = "%sgene_subnum.csv"%dlkcat_folder
sub_description_path = '%sget_gene_subunitDescription.csv'%dlkcat_folder
inchikey_list_file='%sinchikey_list.csv'%dlkcat_folder
inchikey_list_smilesfile='%sinchikey_list_smiles.csv'%dlkcat_folder
comdf_file= '%scomdf.csv'%dlkcat_folder
metdf_outfile='%smetabolites_reactions_gpr_similes_prosequence_mass_dropna.csv'%dlkcat_folder
metabolites_reactions_gpr_file = '%smetabolites_reactions_gpr.csv'%dlkcat_folder
prodf_file = '%sprodf.csv'%dlkcat_folder
DLinput_file= '%sDLinput.tsv'%dlkcat_folder

Path exists


In [4]:
DLouputdf_file = 'get_kcat_mw_by_DLkcat/DLoutput.tsv'
DL_reaction_kact_mw_file='./get_kcat_mw_by_DLkcat/reaction_kcat_MW.csv'

## 1.2 read GEM

In [5]:
if re.search(r'\.xml',sbml_path):
    model = cobra.io.read_sbml_model(sbml_path)
elif re.search(r'\.json',sbml_path):
    model = cobra.io.json.load_json_model(sbml_path)

No objective coefficients in model. Unclear what should be optimized


## 1.3 subunit number of each reaction

In [6]:
starttime=datetime.datetime.now()

print("Starting to fetch subunit number of each enzyme")
# get_gene_subunitDescription(sub_description_path,model)#Download from the UniProt API, run it once.
subbnumdf = get_subunit_number(sub_description_path,gene_subnum_path)
print("Calculation done!")
print()

endtime=datetime.datetime.now()
print(endtime-starttime)

Starting to fetch subunit number of each enzyme
Calculation done!

0:00:00.054523


## 1.4 convert metbolites bigg id to smiles（手动替换）

In [7]:
# starttime=datetime.datetime.now()
# 
# print("Starting to convert metbolites bigg id to smiles...")
# metdf_name = get_met_bigg_id(model)
# # inchkeydf = convert_bigg_met_to_inchikey(metdf_name['met'],inchikey_list_file)#from BIGG
# inchkeydf = pd.read_csv('./data/inchikey_list.csv')
# smilesdf = convert_inchikey_to_smiles(inchkeydf,inchikey_list_smilesfile)#from pubchem
# print("Converting done!")
# print()
# 
# endtime=datetime.datetime.now()
# print(endtime-starttime)

## 1.5 get protein sequence and mass in model

In [8]:
starttime=datetime.datetime.now()

print("Starting to get protein sequence and mass in model...")
subbnumdf = pd.read_csv(gene_subnum_path)
prodf = get_model_protein_sequence_and_mass_zxy(model,subbnumdf,prodf_file,sequence_file)
print("Getting done!")
print()
endtime=datetime.datetime.now()
print(endtime-starttime)

Starting to get protein sequence and mass in model...
列名: ['UniProtID', 'Unnamed: 0', 'proteinNames', 'txt_subunit', 'geneNames', 'subunitnumber']
数据前5行:
  UniProtID  Unnamed: 0                                       proteinNames  \
0    Q8NUD8           0  ['Chromosomal replication initiator protein Dn...   
1    Q8NUD6           1                             ['Beta sliding clamp']   
2    Q8NUC6           2                           ['DNA gyrase subunit A']   
3    Q8NTF8           3  ['Sugar transferases involved in lipopolysacch...   
4    Q8NUB2           4    ['ABC-type transporter, periplasmic component']   

                                         txt_subunit  \
0  Oligomerizes as a right-handed, spiral filamen...   
1  Forms a ring-shaped head-to-tail homodimer aro...   
2  Heterotetramer, composed of two GyrA and two G...   
3  Heterotetramer, composed of two GyrA and two G...   
4  Heterotetramer, composed of two GyrA and two G...   

                       geneNames  subuni

## 1.6 split the substrate of reactions to match the gene

In [9]:
starttime=datetime.datetime.now()

print("Starting to split the substrate of reactions to match the gene...")
spdf = split_substrate_to_match_gene(model,metabolites_reactions_gpr_file)
print("Splitting done!")
print()

endtime=datetime.datetime.now()
print(endtime-starttime)

Starting to split the substrate of reactions to match the gene...
Splitting done!

0:00:01.756318


## 1.7 combine the reaction--substrate--gene--protein_sequnce--mass and formate DLKcat input file

In [10]:
starttime=datetime.datetime.now()

print("Starting to combine data...")
metdf_name = get_met_bigg_id(model)
smilesdf = pd.read_csv(inchikey_list_smilesfile)
spdf = pd.read_csv(metabolites_reactions_gpr_file)
prodf = pd.read_csv(prodf_file)

comdf = combine_reactions_smiles_sequence(spdf,smilesdf,prodf,comdf_file)
DLinputdf = generate_DLKCAT_input(comdf,metdf_name,metdf_outfile,DLinput_file)
print("Combinning done!")
print()

endtime=datetime.datetime.now()
print(endtime-starttime)

Starting to combine data...
Index(['reactions', 'metabolites', 'metabolitestotal', 'gpr', 'genes',
       'similes', 'prosequence', 'mass', 'totalmass', 'metname'],
      dtype='object')
DLKCAT input file generated
Combinning done!

0:00:02.471465


## 1.8 use DLKcat calculate kcat

In [11]:
starttime=datetime.datetime.now()

print("Starting to Use DLKcat calculate kcat...")
cmd_str = "python ./script/prediction_for_input.py ./get_kcat_mw_by_DLkcat/DLinput.tsv ./get_kcat_mw_by_DLkcat/DLoutput.tsv"
subprocess.run(cmd_str, shell=True)
print("DLKcat done!")
print()
endtime=datetime.datetime.now()
print(endtime-starttime)

Starting to Use DLKcat calculate kcat...
DLKcat done!

0:00:15.443237


## 1.9 get the kcat_mw file

In [18]:
starttime=datetime.datetime.now()
# Step 7: get the kcat_mw file
print("Starting to get reaction kcat_mw for model......")
DLouputdf = pd.read_csv(DLouputdf_file, sep='\t')
comdf = pd.read_csv(comdf_file)
DL_reaction_kact_mw = DL_kcat_mw_calculation(DLouputdf, comdf, method="catapro")
DL_reaction_kact_mw.to_csv(DL_reaction_kact_mw_file, index=False)
print("Reaction kcat_mw done!")
endtime=datetime.datetime.now()
print(endtime-starttime)

Starting to get reaction kcat_mw for model......
Successfully generated DL_reaction_kact_mw with 2148 entries
Reaction kcat_mw done!
0:00:00.070076
